# Stochastic gradient descent (with and without pytorch)

## Problem formulation and sampled data

We will import modules as we need them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Create a random number generator (specify the seed if you want repeatable results).

In [ ]:
rng = np.random.default_rng(None)

Sample
$$x_i \sim \mathcal{U}\left(0, 5\right)$$
and
$$y_i \sim \mathcal{N}\left( a x_i + b, \sigma^2 \right)$$
for
$$i\in\{1, \dotsc, n\}.$$

In [ ]:
# Parameters
a = 0.35
b = -1.0
sigma = 0.25
n = 5000

# Sampled data
x = rng.uniform(low=0., high=5., size=n)
y = rng.normal(loc=(a * x + b), scale=sigma)

Plot sampled data.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4), tight_layout=True)
ax.plot(x, y, '.', markersize=0.5)
ax.grid()
ax.set_xlim([0., 5.])
ax.set_ylim([-1.5, 1.5])
ax.set_aspect('equal')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$', rotation='horizontal')
plt.show()

## Gradient descent (with backtracking line search)

Suppose we want to estimate the parameters $\theta = (a, b)$ given only the sampled data. Here is how we might do that by gradient descent with backtracking line search.

In [ ]:
# Define parameters
alpha = 1e-1
max_iters = 250
max_inner_iters = 25
c = 0.1
rho = 0.5
tol = 1e-4

# Choose initial guess
theta = np.zeros(2)

# Create log
log = {
    'theta': [theta.copy()],
    'mse': [],
    'grad_norm': [],
}

# Repeat until convergence or failure
success = False
for iter in range(max_iters):

    # Descent direction (gradient)
    batch_mse = 0.
    batch_grad = np.zeros(2)
    for x_i, y_i in zip(x, y):
        res = ((theta[0] * x_i + theta[1]) - y_i)   # <-- compute signed residual
        batch_mse += np.linalg.norm(res)**2         # <-- update mse
        batch_grad += res * np.array([x_i, 1.])    # <-- update gradient
    batch_grad /= n     # <-- normalize gradient (means we minimize MSE not SSE)
    batch_mse /= n      # <-- normalize error
    batch_grad_norm = np.linalg.norm(batch_grad, ord=np.inf)

    # Backtracking line search
    alpha = 1.
    p = - batch_grad.copy()
    no_progress = True
    for i_inner in range(max_inner_iters):
        theta_new = theta + alpha * p
        mse_new = np.sum(((theta_new[0] * x + theta_new[1]) - y)**2) / n
        if mse_new <= batch_mse + c * alpha * np.dot(batch_grad, p):
            no_progress = False
            break
        else:
            alpha *= rho
    
    # Check for no progress
    if no_progress:
        break

    # Do a gradient descent step
    theta -= alpha * batch_grad

    # Update log
    log['theta'].append(theta.copy())
    log['mse'].append(batch_mse)
    log['grad_norm'].append(batch_grad_norm)

    # Check for convergence
    if batch_grad_norm < tol:
        success = True
        break

# Check for success
if success:
    print('success')
elif no_progress:
    print('failure (no progress in backtracking line search)')
else:
    print('failure (exceeded maximum number of iterations)')

# Clean up log
for k in log.keys():
    log[k] = np.array(log[k])

Plot convergence.

In [ ]:
fig, (ax_theta, ax_mse) = plt.subplots(2, 1, figsize=(6, 5), tight_layout=True, sharex=True)
ax_theta.plot(log['theta'][:, 0], label=r'$\widehat{a}$', linewidth=2)
ax_theta.plot(log['theta'][:, 1], label=r'$\widehat{b}$', linewidth=2)
ax_theta.plot(a * np.ones(len(log['theta'])), '--', color='C0', linewidth=1)
ax_theta.plot(b * np.ones(len(log['theta'])), '--', color='C1', linewidth=1)
ax_theta.grid()
ax_theta.legend()
ax_theta.set_ylabel('Parameter Values')
ax_mse.plot(log['mse'], label='MSE', linewidth=2)
ax_mse.set_xlabel('Iteration')
ax_mse.set_ylabel('Mean Square Error')
ax_mse.legend()
ax_mse.grid()
ax_mse.set_xlim(0, iter)
plt.show()

## Stochastic gradient descent (with fixed step size)

**Stochastic gradient descent** is just gradient descent with a fixed step size but using only a small subset of sampled data (chosen uniformly at random) at each iteration. The subset is called a **minibatch**.

In [ ]:
# Define parameters
batch_size = 32
alpha = 1e-3
max_iters = 25000

# Choose initial guess
theta = np.zeros(2)

# Create log
log = {
    'theta': [theta.copy()],
    'mse': [],
}

# Do a fixed number of SGD iterations
for iter in range(max_iters):

    # Sample a minibatch
    batch_index = rng.choice(len(x), size=batch_size, replace=False)

    # Do one step of gradient descent
    batch_grad = np.zeros(2)
    batch_mse = 0.
    for i in batch_index:
        res = ((theta[0] * x[i] + theta[1]) - y[i]) # <-- compute signed residual
        batch_mse += np.linalg.norm(res)**2         # <-- update mse
        batch_grad += res * np.array([x[i], 1.])    # <-- update gradient
    batch_grad /= batch_size # <-- normalize gradient (means we minimize MSE not SSE)
    batch_mse /= batch_size  # <-- normalize error
    theta -= alpha * batch_grad

    # Update log
    log['theta'].append(theta.copy())
    log['mse'].append(batch_mse)

# Clean up log
for k in log.keys():
    log[k] = np.array(log[k])

Import a module that makes it easy to smooth data.

In [ ]:
import pandas as pd

Plot convergence. This is a standard way to check if you ran SGD for enough iterations. Did you notice that we ran SGD for a fixed number of iterations in the code above? It is not so clear how to define a stopping condition in this case.

In [ ]:
# Choose a smoothing factor (this value matches the tensorboard default)
smooth_factor = 0.99

# Extract and smooth the mean square error
mse = log['mse']
mse_smoothed = pd.Series(mse).ewm(alpha=(1 - smooth_factor)).mean().to_numpy()

# Plot the results
fig, (ax_theta, ax_mse) = plt.subplots(2, 1, figsize=(6, 5), tight_layout=True, sharex=True)
ax_theta.plot(log['theta'][:, 0], label=r'$\widehat{a}$', linewidth=2)
ax_theta.plot(log['theta'][:, 1], label=r'$\widehat{b}$', linewidth=2)
ax_theta.plot(a * np.ones(len(log['theta'])), '--', color='C0', linewidth=1)
ax_theta.plot(b * np.ones(len(log['theta'])), '--', color='C1', linewidth=1)
ax_theta.grid()
ax_theta.legend()
ax_theta.set_ylabel('Parameter Values')
ax_mse.plot(mse, alpha=0.25, color='C0')
ax_mse.plot(mse_smoothed, color='C0', label=f'MSE (smoothed)')
ax_mse.set_xlabel('Iteration')
ax_mse.set_ylabel('Mean Square Error')
ax_mse.legend()
ax_mse.grid()
ax_mse.set_xlim(0, iter)
plt.show()

## Stochastic gradient descent (with Adam optimizer and pytorch)

Import modules for pytorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

Pytorch is a library for deep learning. If we define our model and our loss function using pytorch, then we can use pytorch's implementation of stochastic gradient descent instead of our own. While pytorch has an implementation of SGD (`optim.SGD`), almost everyone uses the "Adam optimizer" instead (`optim.Adam`) - think of this as having a very clever way to choose an adaptive step size.

In [ ]:
# Convert data to tensors
x_tensor = torch.tensor(x, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Define parameters
batch_size = 32
max_iters = 5000

# Define model and choose initial guess at parameter values
model = nn.Linear(
    in_features=1,  # has one input
    out_features=1, # has one output
    bias=True,      # has a bias term
)
nn.init.zeros_(model.weight)
nn.init.zeros_(model.bias)

# Define loss function (mean square error)
loss_fn = nn.MSELoss()

# Define optimizer
optimizer = optim.Adam(model.parameters())

# Create log
log = {
    'theta': [np.array([model.weight.item(), model.bias.item()])],
    'mse': [],
}

# Do a fixed number of Adam iterations
for iter in range(max_iters):

    # Sample a minibatch
    batch_index = rng.choice(len(x), size=batch_size, replace=False)
    x_batch = x_tensor[batch_index].unsqueeze(1) # <-- need to "unsqueeze" because model
                                                 #     expects a tensor of size
                                                 #      (batch_size, 1)
                                                 #     and not
                                                 #      (batch_size,).
    y_batch = y_tensor[batch_index]

    # Do one step
    # - Reset the gradient to zero
    optimizer.zero_grad()
    # - Do forward pass (get predictions from model) - need to "squeeze" because
    #   loss_fn expects a tensor of size (batch_size,) and not (batch_size, 1)
    y_pred = model(x_batch).squeeze()
    # - Compute mean-square-error between predictions and targets (this
    #   is the loss function we want to minimize)
    loss = loss_fn(y_pred, y_batch)
    # - Do backward pass (compute the gradient of the loss function with
    #   respect to all model parameters)
    loss.backward()
    # - Update the model parameters by taking one step with the Adam
    #   optimizer (it is a more sophisticated version of SGD that, more
    #   or less, uses an adaptive learning rate)
    optimizer.step()

    # Update log
    log['mse'].append(loss.item())
    log['theta'].append(np.array([model.weight.item(), model.bias.item()]))

# Clean up log
for k in log.keys():
    log[k] = np.array(log[k])

Plot convergence. Did we run the optimizer for enough iterations?

In [ ]:
# Choose a smoothing factor (this value matches the tensorboard default)
smooth_factor = 0.99

# Extract and smooth the mean square error
mse = log['mse']
mse_smoothed = pd.Series(mse).ewm(alpha=(1 - smooth_factor)).mean().to_numpy()

# Plot the results
fig, (ax_theta, ax_mse) = plt.subplots(2, 1, figsize=(6, 5), tight_layout=True, sharex=True)
ax_theta.plot(log['theta'][:, 0], label=r'$\widehat{a}$', linewidth=2)
ax_theta.plot(log['theta'][:, 1], label=r'$\widehat{b}$', linewidth=2)
ax_theta.plot(a * np.ones(len(log['theta'])), '--', color='C0', linewidth=1)
ax_theta.plot(b * np.ones(len(log['theta'])), '--', color='C1', linewidth=1)
ax_theta.grid()
ax_theta.legend()
ax_theta.set_ylabel('Parameter Values')
ax_mse.plot(mse, alpha=0.25, color='C0')
ax_mse.plot(mse_smoothed, color='C0', label=f'MSE (smoothed)')
ax_mse.set_xlabel('Iteration')
ax_mse.set_ylabel('Mean Square Error')
ax_mse.legend()
ax_mse.grid()
ax_mse.set_xlim(0, iter)
plt.show()

## Same problem but solved with a much more complex model

Here, rather than use a linear model (which assumes we know the structure of our data), we use a deep neural network with one hidden layer and with rectified linear unit activation functions. This is the same sort of neural network you might use to represent a value function or a policy for DQN, DDPG, PPO, etc. Other than changing the model, everything else is exactly the same as before.

In [ ]:
# Convert data to tensors
x_tensor = torch.tensor(x, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Define parameters
batch_size = 32
max_iters = 1000

################
# MODIFIED
#
# Define model (with Kaiming uniform / He initialization)
model = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
).float()   # <-- the type of parameters in the model should
            #     match the type of values in the data
#
################

# Define loss function (mean square error)
loss_fn = nn.MSELoss()

# Define optimizer
optimizer = optim.Adam(model.parameters())

# Create log
log = {
    'mse': [],
}

# Do a fixed number of Adam iterations
for iter in range(max_iters):

    # Sample a minibatch
    batch_index = rng.choice(len(x), size=batch_size, replace=False)
    x_batch = x_tensor[batch_index].unsqueeze(1)
    y_batch = y_tensor[batch_index]

    # Do one step
    optimizer.zero_grad()
    y_pred = model(x_batch).squeeze()
    loss = loss_fn(y_pred, y_batch)
    loss.backward()
    optimizer.step()

    # Update log
    log['mse'].append(loss.item())

# Clean up log
for k in log.keys():
    log[k] = np.array(log[k])

Plot convergence. Also, compare the true and predicted relationship between $x$ and $y$. (We can't compare the true and predicted values of $a$ and $b$ anymore, because the learned model doesn't have those parameters.)

In [ ]:
# Choose a smoothing factor (this value matches the tensorboard default)
smooth_factor = 0.99

# Extract and smooth the mean square error
mse = log['mse']
mse_smoothed = pd.Series(mse).ewm(alpha=(1 - smooth_factor)).mean().to_numpy()

# Plot the results
fig, (ax_mse, ax_xy) = plt.subplots(2, 1, figsize=(6, 5), tight_layout=True)
ax_mse.plot(mse, alpha=0.25, color='C0')
ax_mse.plot(mse_smoothed, color='C0', label=f'MSE (smoothed)')
ax_mse.set_xlabel('Iteration')
ax_mse.set_ylabel('Mean Square Error')
ax_mse.legend()
ax_mse.set_xlim(0, iter)
# - Raw data
ax_xy.plot(x, y, '.', markersize=0.25)
# - Predictions from actual model
x_tru = np.linspace(0., 5., 100)
y_tru = a * x_tru + b
ax_xy.plot(x_tru, y_tru, '--', color='C0', label='true')
# - Predictions from learned model
x_est = torch.linspace(x.min(), x.max(), 100).unsqueeze(1)
with torch.no_grad():
    y_est = model(x_est).squeeze()
x_est = x_est.squeeze().numpy()
y_est = y_est.numpy()
ax_xy.plot(x_est, y_est, '-', label='learned')
ax_xy.set_xlabel(r'$x$')
ax_xy.set_ylabel(r'$y$', rotation='horizontal')
ax_xy.legend()
ax_xy.set_xlim(0, 5)
plt.show()

## SGD for nonlinear regression

Sample
$$x_i \sim \mathcal{U}\left(-10, 10\right)$$
and
$$y_i \sim \mathcal{N}\left( x_i \sin(x_i), \sigma^2 \right)$$
for
$$i\in\{1, \dotsc, n\}.$$

In [ ]:
# Parameters
sigma = 0.5
n = 5000
x_min = -10.
x_max = 10.

# Sampled data
x = rng.uniform(low=x_min, high=x_max, size=n)
y = rng.normal(loc=(x * np.sin(x)), scale=sigma)

Plot sampled data.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4), tight_layout=True)
ax.plot(x, y, '.', markersize=0.5)
ax.grid()
ax.set_xlim([x_min, x_max])
ax.set_ylim([-10., 10.])
ax.set_aspect('equal')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$', rotation='horizontal')
plt.show()

Import a model that makes it easy to copy objects.

In [ ]:
import copy

Apply SGD as implemented by the Adam optimizer to train the same neural network (with one hidden layer and with ReLU activation functions) to model the sampled data. This code is *identical* to what we used before, except that we are now storing copies of the model for the purpose of visualization later on.

In [ ]:
# Convert data to tensors
x_tensor = torch.tensor(x, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Define parameters
batch_size = 32
max_iters = 10000
log_interval = 100 # <-- NEW (how many iterations before we log a new copy of the model)

# Define model (with Kaiming uniform / He initialization)
model = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
).float()

# Define loss function (mean square error)
loss_fn = nn.MSELoss()

# Define optimizer
optimizer = optim.Adam(model.parameters())

# Create log
log = {
    'mse': [],
    'models': [copy.deepcopy(model)],
}

# Do a fixed number of Adam iterations
for iter in range(max_iters):

    # Sample a minibatch
    batch_index = rng.choice(len(x), size=batch_size, replace=False)
    x_batch = x_tensor[batch_index].unsqueeze(1)
    y_batch = y_tensor[batch_index]

    # Do one step
    optimizer.zero_grad()
    y_pred = model(x_batch).squeeze()
    loss = loss_fn(y_pred, y_batch)
    loss.backward()
    optimizer.step()

    # Update log
    log['mse'].append(loss.item())
    if (iter + 1) % log_interval == 0:
        log['models'].append(copy.deepcopy(model))

# Clean up log
for k in log.keys():
    if k not in ['models']:
        log[k] = np.array(log[k])

Import modules for animation.

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

Create an animated plot of convergence.

In [ ]:
smooth_factor = 0.99
mse = log['mse']
mse_smoothed = pd.Series(mse).ewm(alpha=(1 - smooth_factor)).mean().to_numpy()

x_tru = np.linspace(x_min, x_max, 100)
y_tru = x_tru * np.sin(x_tru)
x_est_tensor = torch.linspace(x_min, x_max, 100).unsqueeze(1)

fig, (ax_mse, ax_xy) = plt.subplots(2, 1, figsize=(6, 5), tight_layout=True)

# Static elements
ax_mse.plot(mse, alpha=0.25, color='C0')
ax_mse.plot(mse_smoothed, color='C0', label='MSE (smoothed)')
ax_xy.plot(x, y, '.', markersize=0.25)
ax_xy.plot(x_tru, y_tru, '--', color='C0', label='true')

# Animated elements
vline = ax_mse.axvline(0, color='C1', linewidth=2)
line_est, = ax_xy.plot([], [], '-', color='C1', linewidth=2, label='learned')

# Style
ax_mse.set_xlabel('Iteration')
ax_mse.set_ylabel('Mean Square Error')
ax_mse.legend(loc='upper right')
ax_mse.set_xlim([0, iter])
ax_xy.set_xlabel(r'$x$')
ax_xy.set_ylabel(r'$y$', rotation='horizontal')
ax_xy.legend(loc='upper center')
ax_xy.set_xlim([x_min, x_max])

def update(frame):
    # Update vertical line on MSE plot
    vline.set_xdata([frame * log_interval])
    # Update learned curve
    m = log['models'][frame]
    with torch.no_grad():
        y_est = m(x_est_tensor).squeeze().numpy()
    line_est.set_data(x_est_tensor.squeeze().numpy(), y_est)
    return vline, line_est

ani = animation.FuncAnimation(
    fig, update, frames=len(log['models']), interval=50, blit=True
)
plt.close()  # prevent static display
HTML(ani.to_jshtml())

Create a static plot of convergence.

In [ ]:
# Choose a smoothing factor (this value matches the tensorboard default)
smooth_factor = 0.99

# Extract and smooth the mean square error
mse = log['mse']
mse_smoothed = pd.Series(mse).ewm(alpha=(1 - smooth_factor)).mean().to_numpy()

# Plot the results
fig, (ax_mse, ax_xy) = plt.subplots(2, 1, figsize=(6, 5), tight_layout=True)
ax_mse.plot(mse, alpha=0.25, color='C0')
ax_mse.plot(mse_smoothed, color='C0', label=f'MSE (smoothed)')
ax_mse.set_xlabel('Iteration')
ax_mse.set_ylabel('Mean Square Error')
ax_mse.legend(loc='upper right')
ax_mse.set_xlim([0, iter])
# - Raw data
ax_xy.plot(x, y, '.', markersize=0.25)
# - Predictions from actual model
x_tru = np.linspace(x_min, x_max, 100)
y_tru = x_tru * np.sin(x_tru)
ax_xy.plot(x_tru, y_tru, '--', color='C0', label='true')
# - Predictions from learned model
x_est = torch.linspace(x_min, x_max, 100).unsqueeze(1)
with torch.no_grad():
    y_est = model(x_est).squeeze()
x_est = x_est.squeeze().numpy()
y_est = y_est.numpy()
ax_xy.plot(x_est, y_est, '-', label='learned')
ax_xy.set_xlabel(r'$x$')
ax_xy.set_ylabel(r'$y$', rotation='horizontal')
ax_xy.legend(loc='upper center')
ax_xy.set_xlim([x_min, x_max])
plt.show()